# 🔬 Compute Fabric: Google Colab Compatibility & Capability Test Suite

This notebook comprehensively tests the **Compute Fabric** project in a **Google Colab** environment.

> **Project Overview:** Compute Fabric monitors low-level GPU interconnect errors, thermal spikes, and kernel diagnostics (Xid/AER) to forecast hardware failures and trigger cluster isolation commands (Kubernetes cordon / Slurm drain).

---

## 📌 Quick Summary: What Colab CAN vs CANNOT Do

| Feature / Module | Can it run in Colab? | Status & Behavior in Colab |
| :--- | :---: | :--- |
| **Unit Tests (`pytest`)** | ✅ **YES** | All 13 tests run and pass without physical hardware dependencies. |
| **Early Warning Rules (`rules.py`)** | ✅ **YES** | Evaluates sliding windows, ECC error counters, and throttles in-memory. |
| **Cluster Action Formatter (`actions.py`)** | ✅ **YES (Dry-Run)** | Generates and logs `kubectl cordon` & `scontrol update state=drain`. Live execution will fail because Colab is not connected to a Kubernetes/Slurm cluster. |
| **Log Watcher (`logwatch.py`)** | ⚠️ **PARTIAL** | Log regex parsing works 100%. Reading live kernel logs (`dmesg`) is blocked inside unprivileged Colab Docker containers (`dmesg: read kernel buffer failed: Operation not permitted`). |
| **Replay Simulation (`cli.py replay`)** | ✅ **YES** | Fully operational for evaluating pre-recorded or simulated CSV telemetry. |
| **High-Frequency NVML Collector (`collector.py`)** | ⚠️ **PARTIAL (GPU Mode)** | Requires Colab **GPU runtime** (T4/V100/A100). Works for GPU metrics (utilization, clock, temperature, power), but ECC/PCIe link counters are masked or unsupported on cloud virtualized GPUs. |
| **Overhead Benchmark (`bench.py`)** | ⚠️ **PARTIAL (GPU Mode)** | Runs PyTorch CUDA matrix multiplications, but NVML high-frequency polling accuracy in virtualized cloud environments exhibits jitter. |

--- 
## Step 1: Environment Setup & Hardware Verification
Check whether this Colab runtime has a GPU allocated.

In [ ]:
# Check GPU availability via nvidia-smi
!nvidia-smi

In [ ]:
# Verify Python and install project dependencies
!python --version
!pip install nvidia-ml-py pandas matplotlib pytest torch
# If you cloned or uploaded Compute-fabric, install it in editable mode:
# !pip install -e .

--- 
## Step 2: Run Full Unit Test Suite
Validates that all core logic (rules, state changes, log parsing, scheduler commands) works in the Linux Colab container.

In [ ]:
!pytest -v

--- 
## Step 3: Test Replay Mode (Simulation without Hardware Constraints)
Feeds recorded telemetry into the rules engine to test transition from `OK` to `FAILING`.

In [ ]:
from febric.cli import replay_csv

# Run replay on bundled telemetry dataset
replay_csv("data/samples/telemetry_sample.csv", node_name="colab-worker-01", scheduler_type="kubernetes")

--- 
## Step 4: Test Kernel Diagnostic Watcher (`logwatch.py`)
Test parsing real diagnostic events vs Colab kernel dmesg restrictions.

In [ ]:
from febric.logwatch import run_dmesg, parse_log_lines

print("1. Testing direct dmesg execution in Colab container:")
dmesg_output = run_dmesg()
print(f"-> Lines captured from host dmesg: {len(dmesg_output)}")
if len(dmesg_output) == 0:
    print("   [EXPECTED IN COLAB] Colab unprivileged Docker container blocks raw dmesg access.")

print("\n2. Testing event parser on synthetic fault logs (XID / AER):")
with open("data/samples/dmesg_sample.txt", "r") as f:
    sample_lines = f.read().splitlines()

events = parse_log_lines(sample_lines, reset_index=True)
print(f"-> Parsed {len(events)} critical hardware events:")
for ev in events:
    print(f"   - Type: {ev['type']} | Code/Severity: {ev['code_or_severity']}")

--- 
## Step 5: Test NVML Hardware Telemetry Collector (GPU Runtimes only)
Queries NVML directly on the allocated Colab GPU to test metric availability.

In [ ]:
try:
    from febric.collector import init_nvml_gpu, get_gpu_reading
    import json

    handle = init_nvml_gpu()
    reading = get_gpu_reading(handle)
    print("NVML Query Result from Colab GPU:")
    print(json.dumps(reading, indent=2, default=str))
except Exception as e:
    print(f"NVML test unavailable (CPU runtime or driver restriction): {e}")

--- 
## Step 6: Test Actions & State Logging (`actions.py`)
Simulates failure conditions and inspects generated audit trails.

In [ ]:
from febric.rules import GPUHealthState
from febric.actions import handle_gpu_state_action, reset_action_state
from pathlib import Path

reset_action_state()
log_file = "colab_actions.log"

# Transition OK -> WATCH
handle_gpu_state_action(GPUHealthState.WATCH, "colab-node", reason="Thermal spike 87C", log_file=log_file)

# Transition WATCH -> FAILING (triggers isolation command)
cmd = handle_gpu_state_action(GPUHealthState.FAILING, "colab-node", scheduler_type="kubernetes", reason="Xid 79 fallen off bus", dry_run=True, log_file=log_file)

print("\nGenerated Action Command:", cmd)
print("\nAction Log Contents:")
print(Path(log_file).read_text())